# Corpus Knowledge Graph Construction Practice

This notebook expands the beginner knowledge graph construction workflow from one PDF to a small scientific corpus. The goal is still teaching, not production hardening: parse documents, extract entities, normalize them with an ontology, resolve duplicates, write a graph to Neo4j, store chunks in Weaviate, and query the combined graph/vector representation from Jupyter.

The easiest way to run the full demo is with Docker: copy `.env.example` to `.env` at the project root, set `OPENAI_API_KEY`, run `docker compose up --build`, and open Jupyter Lab at `http://localhost:8888`. The notebook also includes deterministic fallback entities so the non-LLM sections remain readable without an API key.


## 1. Setup

Load environment variables, find the notebook root, and import the helper functions used throughout the demo.


In [ ]:
from pathlib import Path
from json import JSONDecodeError as StdlibJSONDecodeError
from urllib.request import urlopen
import json
import os
import sys
from pprint import pprint

from dotenv import load_dotenv
from neo4j import GraphDatabase
from requests import exceptions as requests_exceptions


load_dotenv()

from helper.kg_demo_helpers import *

api_key = os.getenv("OPENAI_API_KEY", "")
HAS_OPENAI_KEY = bool(api_key and not api_key.startswith("sk-your-key"))

print("OpenAI model:", os.getenv("OPENAI_MODEL", "gpt-4.1-mini"))
print("OpenAI key configured:", HAS_OPENAI_KEY)


### Verify database connections

Run this check after creating `.env` on Expanse. The Weaviate readiness endpoint returns `200 OK` with an empty body when it is healthy, and Neo4j should return `1` from a simple Cypher query.


In [ ]:
def verify_weaviate_ready() -> None:
    host = os.getenv("WEAVIATE_HTTP_HOST", "localhost")
    port = os.getenv("WEAVIATE_HTTP_PORT", "18080")
    url = f"http://{host}:{port}/v1/.well-known/ready"
    with urlopen(url, timeout=5) as response:
        if response.status != 200:
            raise RuntimeError(f"Weaviate readiness check returned HTTP {response.status}: {url}")
        print(f"Weaviate ready: HTTP {response.status} at {url}")


def verify_neo4j_ready() -> None:
    uri = os.getenv("NEO4J_URI", "bolt://localhost:7687")
    username = os.getenv("NEO4J_USERNAME", "neo4j")
    password = os.getenv("NEO4J_PASSWORD", "please-change-me")
    driver = GraphDatabase.driver(uri, auth=(username, password))
    with driver:
        with driver.session() as session:
            ok = session.run("RETURN 1 AS ok").single()["ok"]
    print(f"Neo4j ready: RETURN {ok} from {uri}")


verify_weaviate_ready()
verify_neo4j_ready()


## 2. Start with a small corpus

The helper creates three tiny PDF documents about drought planning, hydrology, remote sensing, Weaviate, and Neo4j. To use your own papers, replace `PDF_PATHS` with a list of local PDF paths.


In [ ]:
CORPUS_DIR = ("data/corpus")
PDF_PATHS = prepare_demo_corpus(CORPUS_DIR)

# To use your own files instead, replace the line above with something like:
# PDF_PATHS = [Path("/path/to/paper1.pdf"), Path("/path/to/paper2.pdf")]

for path in PDF_PATHS:
    print(path)


## 3. Parse documents and chunk text

Each PDF becomes a `CorpusDocument`. Each document is split into `ChunkRecord` objects with document metadata preserved on every chunk.


In [ ]:
documents = load_corpus_documents(PDF_PATHS)
chunks = chunk_corpus_documents(documents, max_chars=600, overlap=80)

print(f"Documents loaded: {len(documents)}")
print(f"Chunks created: {len(chunks)}")
for document in documents:
    print(f"- {document.source_name}: {len(document.text)} characters")

pprint([chunk.model_dump() for chunk in chunks[:3]])


## 4. Store corpus chunks in Weaviate

Weaviate stores the document chunks for retrieval. `semantic_search_chunks` will try vector-style `near_text` search when the collection supports it and falls back to keyword retrieval when the local stack is not configured with a vectorizer.


In [ ]:
# Compatibility for Expanse's system requests 2.25.1.
if not hasattr(requests_exceptions, "JSONDecodeError"):
    requests_exceptions.JSONDecodeError = StdlibJSONDecodeError

weaviate_client = connect_weaviate()
try:
    load_summary = load_chunks_into_weaviate(weaviate_client, chunks)
finally:
    weaviate_client.close()

load_summary


## 5. Extract entities and relationships

With an OpenAI API key, this section extracts entities and relationships from each document. Without a key, it uses deterministic fallback extractions that match the demo corpus, so later ontology and graph steps can still be inspected.


In [ ]:
raw_entities = []
raw_relationships = []

if HAS_OPENAI_KEY:
    for document in documents:
        document_entities = extract_entities(document.text)
        raw_entities.extend(
            {**entity.model_dump(), "source_name": document.source_name}
            for entity in document_entities
        )
        document_relationships = extract_relationships(document.text, document_entities)
        raw_relationships.extend(
            {**relationship.model_dump(), "source_name": document.source_name}
            for relationship in document_relationships
        )
else:
    raw_entities = [
        {"name": "SDSC", "type": "Organization", "description": "Research computing organization.", "evidence": "SDSC runs a summer demo.", "source_name": "01_drought_observations.pdf"},
        {"name": "Center for Hydrology", "type": "Organization", "description": "Hydrology research group.", "evidence": "The Center for Hydrology shares sensor data.", "source_name": "01_drought_observations.pdf"},
        {"name": "satellite imagery", "type": "Dataset", "description": "Imagery used to monitor drought conditions.", "evidence": "Researchers use satellite imagery.", "source_name": "01_drought_observations.pdf"},
        {"name": "river sensors", "type": "Dataset", "description": "Sensors measuring river conditions.", "evidence": "Researchers use ... river sensors.", "source_name": "01_drought_observations.pdf"},
        {"name": "drought planning", "type": "Concept", "description": "Planning for drought conditions.", "evidence": "drought planning in California", "source_name": "01_drought_observations.pdf"},
        {"name": "San Diego Supercomputer Center", "type": "Organization", "description": "Research computing organization.", "evidence": "San Diego Supercomputer Center students compare Remote Sensing Imagery.", "source_name": "02_remote_sensing_methods.pdf"},
        {"name": "Remote Sensing Imagery", "type": "Dataset", "description": "Satellite-based observations.", "evidence": "Remote Sensing Imagery", "source_name": "02_remote_sensing_methods.pdf"},
        {"name": "climate reports", "type": "Dataset", "description": "Climate outlook documents.", "evidence": "seasonal climate outlooks", "source_name": "02_remote_sensing_methods.pdf"},
        {"name": "knowledge graph", "type": "Method", "description": "Graph representation of entities and relationships.", "evidence": "The knowledge graph links datasets.", "source_name": "02_remote_sensing_methods.pdf"},
        {"name": "Weaviate", "type": "Method", "description": "Vector database for chunk retrieval.", "evidence": "stores OCR text chunks in Weaviate", "source_name": "03_graph_vector_search.pdf"},
        {"name": "Neo4j", "type": "Method", "description": "Graph database for normalized relationships.", "evidence": "Neo4j stores normalized entities and relationships.", "source_name": "03_graph_vector_search.pdf"},
        {"name": "semantic search", "type": "Method", "description": "Meaning-based retrieval over chunks.", "evidence": "Weaviate for semantic search", "source_name": "03_graph_vector_search.pdf"},
    ]
    raw_relationships = [
        {"source": "SDSC", "target": "satellite imagery", "relationship": "USES", "description": "SDSC uses satellite imagery.", "evidence": "Researchers use satellite imagery.", "source_name": "01_drought_observations.pdf"},
        {"source": "SDSC", "target": "river sensors", "relationship": "USES", "description": "SDSC uses river sensors.", "evidence": "Researchers use ... river sensors.", "source_name": "01_drought_observations.pdf"},
        {"source": "Center for Hydrology", "target": "SDSC", "relationship": "SHARES_WITH", "description": "The center shares sensor data with SDSC.", "evidence": "shares sensor data with SDSC", "source_name": "01_drought_observations.pdf"},
        {"source": "satellite imagery", "target": "drought planning", "relationship": "SUPPORTS", "description": "Imagery supports drought planning.", "evidence": "drought planning in California", "source_name": "01_drought_observations.pdf"},
        {"source": "river sensors", "target": "drought planning", "relationship": "SUPPORTS", "description": "Sensor data supports drought planning.", "evidence": "understand water conditions", "source_name": "01_drought_observations.pdf"},
        {"source": "Remote Sensing Imagery", "target": "drought planning", "relationship": "SUPPORTS", "description": "Remote sensing supports drought planning.", "evidence": "support drought planning", "source_name": "02_remote_sensing_methods.pdf"},
        {"source": "climate reports", "target": "drought planning", "relationship": "SUPPORTS", "description": "Climate reports support drought planning.", "evidence": "seasonal climate outlooks", "source_name": "02_remote_sensing_methods.pdf"},
        {"source": "semantic search", "target": "Remote Sensing Imagery", "relationship": "USES", "description": "Semantic search retrieves imagery-related chunks.", "evidence": "semantic search", "source_name": "03_graph_vector_search.pdf"},
        {"source": "Weaviate", "target": "semantic search", "relationship": "SUPPORTS", "description": "Weaviate supports retrieval over chunks.", "evidence": "Weaviate for semantic search", "source_name": "03_graph_vector_search.pdf"},
        {"source": "Neo4j", "target": "knowledge graph", "relationship": "STORES_IN", "description": "Neo4j stores the graph representation.", "evidence": "Neo4j stores normalized entities and relationships.", "source_name": "03_graph_vector_search.pdf"},
    ]

print(f"Raw entities: {len(raw_entities)}")
print(f"Raw relationships: {len(raw_relationships)}")
pprint(raw_entities[:4])


## 6. Normalize entities with ontology

The lightweight ontology maps aliases to canonical names. For example, `SDSC` and `San Diego Supercomputer Center` normalize to the same canonical entity.


In [1]:
ONTOLOGY_PATH = ("ontology/demo_ontology.yml")
ontology = load_demo_ontology(ONTOLOGY_PATH)
normalized_preview = normalize_entities_with_ontology(raw_entities, ontology)

print("Canonical entities in ontology:", len(ontology["canonical_entities"]))
pprint([entity.model_dump() for entity in normalized_preview[:6]])


NameError: name 'load_demo_ontology' is not defined

## 7. Resolve entities across documents

Entity resolution merges duplicate mentions across documents and builds a lookup table used to canonicalize relationship endpoints.


In [ ]:
resolved_entities, entity_resolution_map = resolve_entities(raw_entities, ontology)
resolved_relationships = resolve_relationships(raw_relationships, entity_resolution_map)

print(f"Resolved entities: {len(resolved_entities)}")
print(f"Resolved relationships: {len(resolved_relationships)}")
pprint([entity.model_dump() for entity in resolved_entities[:5]])
pprint([relationship.model_dump() for relationship in resolved_relationships[:5]])


## 8. Write the resolved graph to Neo4j

Neo4j stores canonical `Entity` nodes, `Document` nodes, `MENTIONS` edges, and normalized entity-to-entity relationships.


In [ ]:
neo4j_driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    auth=(
        os.getenv("NEO4J_USERNAME", "neo4j"),
        os.getenv("NEO4J_PASSWORD", "please-change-me"),
    ),
)
try:
    graph_summary = write_resolved_graph_to_neo4j(
        resolved_entities,
        resolved_relationships,
        documents=documents,
        driver=neo4j_driver,
    )
finally:
    neo4j_driver.close()

graph_summary


## 9. Semantic search over the corpus

This asks Weaviate for chunks related to a natural-language question. In a fully vectorized collection, this is semantic search; in a minimal local setup, the helper falls back to keyword retrieval so the tutorial remains usable.


In [ ]:
SEARCH_QUESTION = "Which datasets are used for drought planning?"

weaviate_client = connect_weaviate()
try:
    retrieved_chunks = semantic_search_chunks(weaviate_client, SEARCH_QUESTION, limit=5)
finally:
    weaviate_client.close()

pprint(retrieved_chunks)


## 10. Hybrid graph/vector query

The stable tutorial path classifies the natural-language question into a known template, runs a Cypher query, retrieves relevant chunks, and returns both forms of evidence.


In [ ]:
neo4j_driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    auth=(
        os.getenv("NEO4J_USERNAME", "neo4j"),
        os.getenv("NEO4J_PASSWORD", "please-change-me"),
    ),
)
weaviate_client = connect_weaviate()
try:
    hybrid_result = template_hybrid_query(
        SEARCH_QUESTION,
        neo4j_driver,
        weaviate_client,
        limit=5,
    )
finally:
    neo4j_driver.close()
    weaviate_client.close()

pprint(hybrid_result)


## 11. Optional LLM answer synthesis

This optional cell asks an LLM to write a short answer using only the graph rows and retrieved chunks. The LLM does not query the databases directly in this tutorial path.


In [ ]:
if HAS_OPENAI_KEY:
    answer = llm_hybrid_query(
        SEARCH_QUESTION,
        hybrid_result["graph_rows"],
        hybrid_result["chunks"],
    )
    print(answer)
else:
    print("Set OPENAI_API_KEY in .env at the project root to run optional LLM answer synthesis.")


## 12. Next steps

For a real project, replace the built-in corpus with your own papers, expand the ontology, and use a production entity-resolution strategy. The lightweight YAML ontology can be replaced by RDF, OWL, or SKOS as long as the loader produces the same canonical-name and alias mapping used by the helper functions. Production systems should also add provenance tracking, confidence scores, richer relationship schemas, and live integration tests for Neo4j, Weaviate, and LLM calls.
